In [1]:
from collections import Counter
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random
from sympy import true

import part1 # to use code from tutorial
import torch

def set_seed(seed):
    """Seeds Python, NumPy (also used by sklearn's saga solver) and PyTorch (weight init, shuffling, dropout)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# Shared Step: Create Datasets with Three Representations

In [2]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 1.0  # fraction of the data to use, 1.0 for all of it

# merge the party orientations into two classes, "Left" and "Right"
# "Centre" has very few samples, so it is discarded
orientation_to_side = {
    "Centre-left to left": "Left",
    "Centre-left": "Left",
    "Centre to centre-left": "Left",
    "Centre to centre-right": "Right",
    "Centre-right": "Right",
    "Right": "Right",
    "Right to far-right": "Right",
    "Far-right": "Right",
}

def merge_orientations(df, label_column=LABEL_COL):
    df = df[df[label_column].isin(orientation_to_side.keys())].copy()
    df[label_column] = df[label_column].map(orientation_to_side)
    return df

# get the dataframe: one dataframe with tokens, lemmas, sentences and embeddings,
# so all three representations are trained and evaluated on the same rows
df = merge_orientations(pd.read_pickle("./embeddings_sampled.pkl"))
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
# repeated cross-validation: each seed gives its own fold split and model initialisation
# (fixed in advance; more runs give the Bayesian correlated t-test more paired differences to work with)
my_seeds = [0, 1, 2]
num_folds = 5

In [3]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [4]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

def sparse_row(counts: dict, size: int) -> csr_matrix:
    """A 1 x size sparse row with counts[index] at each index (only the non-zero entries are stored)."""
    indices = sorted(counts)
    values = np.array([counts[i] for i in indices], dtype=np.float32)
    return csr_matrix((values, indices, [0, len(indices)]), shape=(1, size))

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # sparse row instead of np.zeros(len(vocab)), so the full dataset fits in memory
        indices = {self.vocab[item] for item in x.split() if item in self.vocab}
        return sparse_row({index: 1 for index in indices}, len(self))

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column]
    # only join list columns (tokens/lemmas); str.join on a plain string column would put spaces between characters
    if isinstance(texts.iloc[0], (list, tuple, np.ndarray)):
        texts = texts.str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))


In [5]:
# cross-validation folds from the train rows only (the test set stays held out), using the tutorial's function
def make_cv(column):
    pairs = make_pairs(train_df, column, label_names)
    return [part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed) for seed in my_seeds]

cv_lemmas = make_cv("lemmas")        # for token indicator and bag of words
cv_sentences = make_cv("sentence")   # for the embeddings

In [6]:
# evaluation: accuracy and macro F1, per cross-validation fold and on the held-out test set
# (macro F1 averages the F1 of Left and Right, so the smaller Left class counts as much as Right)
# scores are saved to results/<experiment>/cv_scores.csv and test_scores.csv, so they survive a kernel restart
from collections import OrderedDict
from sklearn.metrics import accuracy_score, f1_score

test_lemmas = make_pairs(test_df, "lemmas", label_names)
test_sentences = make_pairs(test_df, "sentence", label_names)

def add_cv_f1(name, data_splits):
    """Adds the macro F1 of every fold's validation predictions to the runs of experiment `name`, next to "accuracies",
    and saves accuracy / macro F1 per seed and fold to results/<name>/cv_scores.csv."""
    rows = []
    for seed, run, splits in zip(my_seeds, my_classifiers[name], data_splits):
        run["f1_scores"] = OrderedDict(
            (fold, f1_score([label for _, label in splits[fold]["validation"]], preds, average="macro"))
            for fold, preds in run["predictions"].items()
        )
        for fold in run["f1_scores"]:
            rows.append({"seed": seed, "fold": fold, "accuracy": run["accuracies"][fold], "f1": run["f1_scores"][fold]})
            print(f"seed {seed} fold {fold}: accuracy {run['accuracies'][fold]:.4f} | macro F1 {run['f1_scores'][fold]:.4f}")
    pd.DataFrame(rows).to_csv(results_folder / name / "cv_scores.csv", index=False)

def load_scores(kind="cv"):
    """All saved scores of one kind ("cv" or "test"), one row per (model, seed, fold)."""
    frames = [pd.read_csv(path).assign(model=path.parent.name) for path in sorted(results_folder.glob(f"*/{kind}_scores.csv"))]
    return pd.concat(frames, ignore_index=True)[["model", "seed", "fold", "accuracy", "f1"]]

def cv_data_frame(metric="f1"):
    """One row per (model, seed, fold) with the saved cross-validation score ("accuracy" or "f1"), e.g. for the ROPE analysis."""
    return load_scores("cv").rename(columns={metric: "score"})[["model", "seed", "fold", "score"]]

def evaluate_on_test(name, test_pairs, encoder, model_option):
    """Loads every fold model of experiment `name`, predicts the test set and saves accuracy / macro F1 per model
    to results/<name>/test_scores.csv."""
    texts = [text for text, _ in test_pairs]
    gold = [label for _, label in test_pairs]
    rows = []
    for seed, run in zip(my_seeds, my_classifiers[name]):
        for fold, path in run["models"].items():
            if model_option == "logistic":
                preds = part1.loglin.predict(texts, part1.loglin.load(path), encoder)
            else:
                preds = part1.smlp.predict_probs(texts, part1.smlp.load(path), encoder, batch_size=512).argmax(axis=1)
            rows.append({"seed": seed, "fold": fold,
                         "accuracy": accuracy_score(gold, preds),
                         "f1": f1_score(gold, preds, average="macro")})
    scores = pd.DataFrame(rows)
    scores.to_csv(results_folder / name / "test_scores.csv", index=False)
    print(scores.set_index(["seed", "fold"]).round(4).to_string())
    print(f"test accuracy {scores.accuracy.mean():.4f} ± {scores.accuracy.std():.4f} | "
          f"test macro F1 {scores.f1.mean():.4f} ± {scores.f1.std():.4f}")
    return scores

In [7]:
# vocabulary from the train rows only (lemmas)
train_texts = [text for text, _ in make_pairs(train_df, "lemmas", label_names)]
vocab = get_vocab(train_texts, min_count=5, max_size=5000)

In [8]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # repeated words are counted; sparse row so the full dataset fits in memory
        counts = Counter(self.vocab[item] for item in x.split() if item in self.vocab)
        return sparse_row(counts, len(self))

In [9]:
class Embedding(Encoder):
    dense = True  # tells part1 to use dense matrices (and lbfgs for logistic regression) instead of sparse ones

    def __init__(self, embeddings_map: pd.DataFrame):
        # build a sentence -> vector dict once, so each lookup is O(1) instead of scanning the whole dataframe
        self.lookup = {
            sentence: np.asarray(embedding, dtype=np.float32)
            for sentence, embedding in zip(embeddings_map['sentence'], embeddings_map['embeddings'])
        }
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.lookup[x]

In [10]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(df)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 3: Two-Hidden-Layer MLP
Backup alternative method: the Experiment 2 MLP with a second hidden layer of the same size.
Same hyperparameters as the matching Experiment 2 run (not tuned separately), so the only change is the extra layer.

Exp 3.A: Two-Layer MLP with Token Indicator

In [11]:
my_classifiers["mlp2_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp2_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp2_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            # Exp 2 settings, with two hidden layers of hidden_dim units
            smlp_args={
                "hidden_dim": 64,
                "num_hidden_layers": 2,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 3e-4,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp2_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...


Document transformation done! Time taken: 5.679 seconds


Dataloader done! Time taken: 0.777 seconds


Training...


  0%|          | 0/25 [00:00<?, ?it/s]

... (418 outputs left out)

Epoch 22 | train loss: 0.3128 | train acc: 0.8528 | val loss: 0.7878 | val acc: 0.6789


Epoch 23 | train loss: 0.3094 | train acc: 0.8544 | val loss: 0.8077 | val acc: 0.6793


Epoch 24 | train loss: 0.3054 | train acc: 0.8562 | val loss: 0.8192 | val acc: 0.6776


Epoch 25 | train loss: 0.3020 | train acc: 0.8570 | val loss: 0.8334 | val acc: 0.6773
Training done! Time taken: 304.139 seconds
fold 0: accuracy 0.7031
fold 1: accuracy 0.6994
fold 2: accuracy 0.7013
fold 3: accuracy 0.7003
fold 4: accuracy 0.6995


In [12]:
add_cv_f1("mlp2_indicator-lemmas", cv_lemmas)
evaluate_on_test("mlp2_indicator-lemmas", test_lemmas, token_ind_encoder, "mlp");

seed 0 fold 0: accuracy 0.7031 | macro F1 0.6736
seed 0 fold 1: accuracy 0.6994 | macro F1 0.6677
seed 0 fold 2: accuracy 0.7013 | macro F1 0.6688
seed 0 fold 3: accuracy 0.7003 | macro F1 0.6652
seed 0 fold 4: accuracy 0.6995 | macro F1 0.6740


seed 1 fold 0: accuracy 0.6993 | macro F1 0.6601
seed 1 fold 1: accuracy 0.7009 | macro F1 0.6640
seed 1 fold 2: accuracy 0.7043 | macro F1 0.6726
seed 1 fold 3: accuracy 0.7000 | macro F1 0.6733
seed 1 fold 4: accuracy 0.7009 | macro F1 0.6662
seed 2 fold 0: accuracy 0.7007 | macro F1 0.6741
seed 2 fold 1: accuracy 0.6995 | macro F1 0.6703
seed 2 fold 2: accuracy 0.7013 | macro F1 0.6670
seed 2 fold 3: accuracy 0.6987 | macro F1 0.6625
seed 2 fold 4: accuracy 0.7029 | macro F1 0.6644


           accuracy      f1
seed fold                  
0    0       0.6994  0.6687
     1       0.6996  0.6675
     2       0.6969  0.6632
     3       0.6966  0.6604
     4       0.6985  0.6731
1    0       0.6974  0.6578
     1       0.6992  0.6625
     2       0.6990  0.6662
     3       0.6957  0.6680
     4       0.6996  0.6645
2    0       0.6975  0.6693
     1       0.7000  0.6690
     2       0.6996  0.6652
     3       0.6993  0.6628
     4       0.7001  0.6625
test accuracy 0.6986 ± 0.0014 | test macro F1 0.6654 ± 0.0040


Exp 3.B: Two-Layer MLP with Bag of Words

In [13]:
my_classifiers["mlp2_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp2_bow-lemmas/seed-{seed}"
    my_classifiers["mlp2_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            # Exp 2 settings, with two hidden layers of hidden_dim units
            smlp_args={
                "hidden_dim": 64,
                "num_hidden_layers": 2,
                "dropout": 0.3,
                "batch_size": 128,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp2_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...


Document transformation done! Time taken: 5.227 seconds


Dataloader done! Time taken: 1.351 seconds
Training...


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.5852 | train acc: 0.6790 | val loss: 0.5628 | val acc: 0.6966


... (416 outputs left out)

Epoch 22 | train loss: 0.3293 | train acc: 0.8459 | val loss: 0.8013 | val acc: 0.6798


Epoch 23 | train loss: 0.3261 | train acc: 0.8473 | val loss: 0.8285 | val acc: 0.6807


Epoch 24 | train loss: 0.3219 | train acc: 0.8489 | val loss: 0.8155 | val acc: 0.6796


Epoch 25 | train loss: 0.3199 | train acc: 0.8497 | val loss: 0.8474 | val acc: 0.6787
Training done! Time taken: 304.946 seconds
fold 0: accuracy 0.7026
fold 1: accuracy 0.6986
fold 2: accuracy 0.7014
fold 3: accuracy 0.7023
fold 4: accuracy 0.7008


In [14]:
add_cv_f1("mlp2_bow-lemmas", cv_lemmas)
evaluate_on_test("mlp2_bow-lemmas", test_lemmas, bow_encoder, "mlp");

seed 0 fold 0: accuracy 0.7026 | macro F1 0.6659
seed 0 fold 1: accuracy 0.6986 | macro F1 0.6620
seed 0 fold 2: accuracy 0.7014 | macro F1 0.6624
seed 0 fold 3: accuracy 0.7023 | macro F1 0.6716
seed 0 fold 4: accuracy 0.7008 | macro F1 0.6718


seed 1 fold 0: accuracy 0.7001 | macro F1 0.6637
seed 1 fold 1: accuracy 0.6990 | macro F1 0.6632
seed 1 fold 2: accuracy 0.7042 | macro F1 0.6689
seed 1 fold 3: accuracy 0.7002 | macro F1 0.6600
seed 1 fold 4: accuracy 0.6993 | macro F1 0.6567
seed 2 fold 0: accuracy 0.7000 | macro F1 0.6653
seed 2 fold 1: accuracy 0.7013 | macro F1 0.6678
seed 2 fold 2: accuracy 0.7014 | macro F1 0.6646
seed 2 fold 3: accuracy 0.6983 | macro F1 0.6540
seed 2 fold 4: accuracy 0.7040 | macro F1 0.6669


           accuracy      f1
seed fold                  
0    0       0.6996  0.6619
     1       0.6999  0.6632
     2       0.6986  0.6590
     3       0.7022  0.6708
     4       0.7002  0.6714
1    0       0.6994  0.6635
     1       0.6979  0.6618
     2       0.6978  0.6617
     3       0.6993  0.6582
     4       0.7007  0.6580
2    0       0.7012  0.6657
     1       0.6995  0.6647
     2       0.7004  0.6631
     3       0.6985  0.6546
     4       0.7014  0.6652
test accuracy 0.6998 ± 0.0013 | test macro F1 0.6629 ± 0.0045


Exp 3.C: Two-Layer MLP with Sentence Embeddings

In [15]:
my_classifiers["mlp2_embeddings"] = []

for run, seed in enumerate(my_seeds):
    set_seed(seed)
    run_folder = f"{results_folder}/mlp2_embeddings/seed-{seed}"
    my_classifiers["mlp2_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            # Exp 2 settings, with two hidden layers of hidden_dim units
            smlp_args={
                "hidden_dim": 128,
                "num_hidden_layers": 2,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp2_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.198 seconds


Dataloader done! Time taken: 1.576 seconds
Training...


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6111 | train acc: 0.6534 | val loss: 0.5937 | val acc: 0.6714


Epoch 02 | train loss: 0.5869 | train acc: 0.6743 | val loss: 0.5857 | val acc: 0.6767


... (402 outputs left out)

Epoch 22 | train loss: 0.4615 | train acc: 0.7730 | val loss: 0.6229 | val acc: 0.6755


Epoch 23 | train loss: 0.4579 | train acc: 0.7759 | val loss: 0.6289 | val acc: 0.6753


Epoch 24 | train loss: 0.4544 | train acc: 0.7775 | val loss: 0.6292 | val acc: 0.6791


Epoch 25 | train loss: 0.4521 | train acc: 0.7795 | val loss: 0.6393 | val acc: 0.6776
Training done! Time taken: 41.879 seconds
fold 0: accuracy 0.6874
fold 1: accuracy 0.6846
fold 2: accuracy 0.6824
fold 3: accuracy 0.6840
fold 4: accuracy 0.6851


In [16]:
add_cv_f1("mlp2_embeddings", cv_sentences)
evaluate_on_test("mlp2_embeddings", test_sentences, sentence_em_encoder, "mlp");

seed 0 fold 0: accuracy 0.6874 | macro F1 0.6392
seed 0 fold 1: accuracy 0.6846 | macro F1 0.6560
seed 0 fold 2: accuracy 0.6824 | macro F1 0.6483
seed 0 fold 3: accuracy 0.6840 | macro F1 0.6526
seed 0 fold 4: accuracy 0.6851 | macro F1 0.6454
seed 1 fold 0: accuracy 0.6850 | macro F1 0.6480
seed 1 fold 1: accuracy 0.6874 | macro F1 0.6467
seed 1 fold 2: accuracy 0.6833 | macro F1 0.6476
seed 1 fold 3: accuracy 0.6802 | macro F1 0.6451
seed 1 fold 4: accuracy 0.6854 | macro F1 0.6451


seed 2 fold 0: accuracy 0.6855 | macro F1 0.6521
seed 2 fold 1: accuracy 0.6855 | macro F1 0.6574
seed 2 fold 2: accuracy 0.6856 | macro F1 0.6539
seed 2 fold 3: accuracy 0.6855 | macro F1 0.6523
seed 2 fold 4: accuracy 0.6856 | macro F1 0.6472


           accuracy      f1
seed fold                  
0    0       0.6816  0.6322
     1       0.6837  0.6537
     2       0.6854  0.6512
     3       0.6799  0.6475
     4       0.6826  0.6437
1    0       0.6815  0.6440
     1       0.6846  0.6430
     2       0.6835  0.6482
     3       0.6827  0.6467
     4       0.6824  0.6409
2    0       0.6826  0.6476
     1       0.6851  0.6559
     2       0.6827  0.6508
     3       0.6847  0.6511
     4       0.6831  0.6452
test accuracy 0.6831 ± 0.0015 | test macro F1 0.6468 ± 0.0058


In [17]:
# summary: cross-validation and test accuracy / macro F1 for every experiment (mean and std over the fold models),
# read from the score files in results/, so this also works after a kernel restart
summary = pd.concat({
    "cv": load_scores("cv").groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]),
    "test": load_scores("test").groupby("model")[["accuracy", "f1"]].agg(["mean", "std"]),
}, axis=1)
majority = test_df[LABEL_COL].value_counts(normalize=True).max()
print(f"majority-class baseline: accuracy {majority:.4f}, macro F1 {majority / (1 + majority):.4f}")
summary.round(4)

majority-class baseline: accuracy 0.6210, macro F1 0.3831


cv                             test          \
                          accuracy              f1         accuracy           
                              mean     std    mean     std     mean     std   
model                                                                         
logistic_bow-lemmas         0.6851  0.0030  0.6323  0.0065   0.6845  0.0024   
logistic_embeddings         0.6569  0.0018  0.5930  0.0019   0.6559  0.0005   
logistic_indicator-lemmas   0.6903  0.0015  0.6466  0.0015   0.6890  0.0007   
mlp2_bow-lemmas             0.7009  0.0018  0.6643  0.0050   0.6998  0.0013   
mlp2_embeddings             0.6848  0.0018  0.6491  0.0048   0.6831  0.0015   
mlp2_indicator-lemmas       0.7008  0.0016  0.6682  0.0046   0.6986  0.0014   
mlp_bow-lemmas              0.7021  0.0015  0.6649  0.0050   0.7005  0.0010   
mlp_embeddings              0.6891  0.0019  0.6538  0.0040   0.6879  0.0012   
mlp_indicator-lemmas        0.7032  0.0021  0.6678  0.0037   0.7008  0.0014   

                                           
                               f1          
                             mean     std  
model                                      
logistic_bow-lemmas        0.6309  0.0059  
logistic_embeddings        0.5911  0.0005  
logistic_indicator-lemmas  0.6443  0.0009  
mlp2_bow-lemmas            0.6629  0.0045  
mlp2_embeddings            0.6468  0.0058  
mlp2_indicator-lemmas      0.6654  0.0040  
mlp_bow-lemmas             0.6627  0.0048  
mlp_embeddings             0.6521  0.0035  
mlp_indicator-lemmas       0.6648  0.0036